# regime_aware_equity_drawdown_guard_v1

Runtime notebook for a fixed-actor execution/risk-mapping loop. The current objective is to keep SOXL excess return above `+30pp`, push backtest max drawdown below `15%`, and avoid selecting a brittle overfit overlay.

## Governance

- **Direction**: `execution`
- **Baselines**: `recovery_reentry_anchor_confirm_v1` and `recovery_reentry_equity_drawdown_guard_v1`
- **Direct comparison**: anchor-confirmed recovery reentry plus global account-equity drawdown governor
- **Role**: `diagnose`
- **Changed layer**: `execution`
- **Target window**: strict OOS `2020-01-02..latest` plus saved walk-forward windows `wf_2020`, `wf_2022`, `wf_2023_2024`
- **Accepted trade-off**: only accept a lower drawdown branch if it keeps `relative_return >= 30pp`; otherwise treat it as underparticipation, not a promotion candidate
- **Hypothesis**: the previous global equity drawdown governor de-risked too broadly. Restricting de-risking to broken-anchor / high-volatility downtrend states may reduce max drawdown without destroying rebound/trend participation.
- **Acceptance criteria**: strict OOS `relative_return >= 30pp` and `max_dd <= 15%`, then the same target checked across saved walk-forward daily paths. A strict-only pass is diagnostic, not promotion.
- **Failure mode**: global controls get below `15%` only by collapsing average position and failing buy-hold-relative return; filtered controls preserve excess but leave max drawdown too high.
- **Comparison-table row key**: `regime_aware_equity_drawdown_guard_v1`

## Historical duplicate check

- **Closest success**: `recovery_reentry_equity_drawdown_guard_v1` proved account-equity drawdown control can reduce strict-OOS MaxDD while preserving more than `+30pp` excess.
- **Closest failure**: global cap tightening and strict-only drawdown guards repeatedly produced low drawdown by underparticipating, then failed rebound/trend windows.
- **Why this is not a duplicate**: this notebook uses the existing optional feature filters in `EquityDrawdownControlConfig` to test regime-aware activation (`anchor_gap_180`, `below_anchor_ratio_63`, `vix_percentile_252`, `vol_of_vol_21`, `mom_21`) instead of another always-on governor.

## External references used

- Qlib GitHub: https://github.com/microsoft/qlib
- Qlib paper: https://arxiv.org/abs/2009.11189
- Deep Momentum Networks: https://arxiv.org/abs/1904.04912
- Drawdown Measure in Portfolio Optimization: https://www.math.columbia.edu/~chekhlov/ChekhlovUryasevZabarankin--03-2004.pdf


In [1]:
from __future__ import annotations

from dataclasses import asdict
import itertools
import json
import shutil
from pathlib import Path
import sys
from typing import Any

from IPython.display import Markdown, display
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "AGENTS.md").exists() and (path / "src" / "soxl_quant").exists():
            return path
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.budget_counterfactual import build_budget_usage_counterfactual
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.policy.two_speed_budget import (
    BullParticipationGuardConfig,
    EquityDrawdownControlConfig,
    RecoveryReentryGuardConfig,
)
from src.soxl_quant.settings import QLIB_DATA_PATH

CARD = "regime_aware_equity_drawdown_guard_v1"
CARD_DIR = ROOT / "runtime" / "execution" / "card" / CARD
CARD_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = CARD_DIR / "selected_artifacts"
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0

CURRENT_INPUTS = {
    "slow": ROOT / "runtime/model/card/v19reg_v22_budget_usage_v1/inputs/balanced_rule_residual_v19_reg_fresh_input.csv",
    "fast": ROOT / "runtime/model/card/v19reg_v22_budget_usage_v1/inputs/balanced_rule_residual_v22_fresh_input.csv",
}
WF_INPUTS = {
    "wf_2020": {
        "slow": ROOT / "runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2020/balanced_rule_residual_v19_reg_input.csv",
        "fast": ROOT / "runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2020/balanced_rule_residual_v22_input.csv",
    },
    "wf_2022": {
        "slow": ROOT / "runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2022/balanced_rule_residual_v19_reg_input.csv",
        "fast": ROOT / "runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2022/balanced_rule_residual_v22_input.csv",
    },
    "wf_2023_2024": {
        "slow": ROOT / "runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2023_2024/balanced_rule_residual_v19_reg_input.csv",
        "fast": ROOT / "runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2023_2024/balanced_rule_residual_v22_input.csv",
    },
}

for label, inputs in {"current": CURRENT_INPUTS, **WF_INPUTS}.items():
    missing = [str(path.relative_to(ROOT)) for path in inputs.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(f"{label} missing input files: {missing}")

print(f"card_dir={CARD_DIR.relative_to(ROOT)}")


card_dir=runtime/execution/card/regime_aware_equity_drawdown_guard_v1


In [2]:
init_qlib(QLIB_DATA_PATH)
dataset = build_dataset()
price = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
features = dataset.handler.feature_frame.droplevel("instrument").sort_index()

bull_guard = BullParticipationGuardConfig(enabled=True)
reentry_guard = RecoveryReentryGuardConfig(
    enabled=True,
    max_below_anchor_ratio_63=0.20,
    min_anchor_gap_180=0.75,
)

print(f"price_range={price.index.min().date()}..{price.index.max().date()} rows={len(price)}")
print(f"feature_columns={len(features.columns)}")


[49168:MainThread](2026-05-24 22:09:31,690) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[49168:MainThread](2026-05-24 22:09:32,486) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[49168:MainThread](2026-05-24 22:09:32,487) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[49168:MainThread](2026-05-24 22:09:35,051) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[49168:MainThread](2026-05-24 22:09:35,053) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[49168:MainThread](2026-05-24 22:09:35,054) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


price_range=2011-03-11..2026-05-22 rows=3823
feature_columns=59


In [3]:
def dd_config_from_spec(spec: dict[str, Any]) -> EquityDrawdownControlConfig:
    if not spec.get("enabled", True):
        return EquityDrawdownControlConfig(enabled=False)
    return EquityDrawdownControlConfig(
        enabled=True,
        start_drawdown=float(spec["start_drawdown"]),
        full_drawdown=float(spec["full_drawdown"]),
        min_target=float(spec["min_target"]),
        sell_trade_threshold=0.0,
        sell_trade_lock_days=0,
        max_anchor_gap_180=spec.get("max_anchor_gap_180"),
        min_below_anchor_ratio_63=spec.get("min_below_anchor_ratio_63"),
        min_vix_percentile_252=spec.get("min_vix_percentile_252"),
        min_vol_of_vol_21=spec.get("min_vol_of_vol_21"),
        max_mom_21=spec.get("max_mom_21"),
    )


def build_result(label: str, inputs: dict[str, Path], spec: dict[str, Any]):
    return build_budget_usage_counterfactual(
        pd.read_csv(inputs["slow"]),
        pd.read_csv(inputs["fast"]),
        price,
        slow_label=spec.get("slow_label", "balanced_rule_residual_v19_reg"),
        fast_label=spec.get("fast_label", "balanced_rule_residual_v22"),
        composed_label=label,
        description=spec.get("description", label),
        profile=label,
        bull_guard=bull_guard,
        feature_frame=features,
        recovery_guard=reentry_guard,
        drawdown_control=dd_config_from_spec(spec),
    )


def metric_row(label: str, result, spec: dict[str, Any], window: str) -> dict[str, Any]:
    metrics = result.payload["variants"][label]["test_proxy"]
    daily = result.daily
    row = {
        "window": window,
        "label": label,
        "filter_family": spec.get("filter_family", "reference"),
        "is_reference": bool(spec.get("is_reference", False)),
        "start_drawdown": spec.get("start_drawdown"),
        "full_drawdown": spec.get("full_drawdown"),
        "min_target": spec.get("min_target"),
        "max_anchor_gap_180": spec.get("max_anchor_gap_180"),
        "min_below_anchor_ratio_63": spec.get("min_below_anchor_ratio_63"),
        "min_vix_percentile_252": spec.get("min_vix_percentile_252"),
        "min_vol_of_vol_21": spec.get("min_vol_of_vol_21"),
        "max_mom_21": spec.get("max_mom_21"),
        "rows": metrics["rows"],
        "start": metrics["start"],
        "end": metrics["end"],
        "return": metrics["return"],
        "buy_and_hold_return": metrics["buy_and_hold_return"],
        "relative_return": metrics["relative_return"],
        "sharpe": metrics["sharpe"],
        "max_dd": metrics["max_dd"],
        "avg_pos": metrics["avg_pos"],
        "stress_avg_pos": metrics["stress_avg_pos"],
        "trades": metrics["trades"],
        "drawdown_control_days": int(daily.get("equity_drawdown_control", pd.Series(dtype=bool)).fillna(False).sum()),
        "drawdown_regime_ready_days": int(daily.get("equity_drawdown_regime_ready", pd.Series(dtype=bool)).fillna(False).sum()),
        "recovery_reentry_days": int(daily.get("recovery_reentry_guard", pd.Series(dtype=bool)).fillna(False).sum()),
    }
    row["relative_gap"] = max(TARGET_RELATIVE_RETURN - row["relative_return"], 0.0)
    row["drawdown_gap"] = max(row["max_dd"] - TARGET_MAX_DD, 0.0)
    row["target_loss"] = row["relative_gap"] + row["drawdown_gap"]
    row["target_pass"] = row["relative_return"] >= TARGET_RELATIVE_RETURN and row["max_dd"] <= TARGET_MAX_DD
    return row


def artifact_safe_label(label: str) -> str:
    return label.replace(".", "p")


def write_selected_artifacts(label: str, result, window: str) -> None:
    prefix = ARTIFACT_DIR / f"{artifact_safe_label(label)}_{window}"
    with Path(f"{prefix}.json").open("w") as f:
        json.dump(result.payload, f, indent=2, allow_nan=False)
    result.daily.to_csv(Path(f"{prefix}.csv"), index=False)


In [4]:
FILTERS = {
    "global": {},
    "anchor_break": {
        "max_anchor_gap_180": 0.0,
        "min_below_anchor_ratio_63": 0.50,
    },
    "panic_break": {
        "max_anchor_gap_180": 0.0,
        "min_below_anchor_ratio_63": 0.50,
        "min_vix_percentile_252": 0.65,
        "min_vol_of_vol_21": 0.08,
    },
    "downtrend_break": {
        "max_anchor_gap_180": 0.0,
        "min_below_anchor_ratio_63": 0.50,
        "max_mom_21": 0.0,
    },
}

START_DRAWDOWNS = [0.04, 0.06, 0.08, 0.10, 0.12]
FULL_DRAWDOWNS = [0.10, 0.14, 0.18, 0.22]
MIN_TARGETS = [0.00, 0.05, 0.10, 0.15, 0.25]

candidate_specs: dict[str, dict[str, Any]] = {}
for family, filter_kwargs in FILTERS.items():
    for start, full, min_target in itertools.product(START_DRAWDOWNS, FULL_DRAWDOWNS, MIN_TARGETS):
        if full <= start:
            continue
        label = f"{family}_s{start:.2f}_f{full:.2f}_m{min_target:.2f}"
        candidate_specs[label] = {
            "enabled": True,
            "filter_family": family,
            "start_drawdown": start,
            "full_drawdown": full,
            "min_target": min_target,
            **filter_kwargs,
            "description": f"{family} regime-aware equity drawdown governor: start={start}, full={full}, min_target={min_target}",
        }

reference_specs = {
    "anchor_confirm_no_dd_control": {
        "enabled": False,
        "filter_family": "anchor_confirm_reference",
        "is_reference": True,
        "description": "Anchor-confirmed recovery reentry without account-equity drawdown control.",
    },
    "global_guard_default": {
        "enabled": True,
        "filter_family": "global_reference",
        "is_reference": True,
        "start_drawdown": 0.12,
        "full_drawdown": 0.22,
        "min_target": 0.25,
        "description": "Previous always-on account-equity drawdown governor reference.",
    },
}

print(f"candidate_specs={len(candidate_specs)} reference_specs={len(reference_specs)}")


candidate_specs=360 reference_specs=2


In [5]:
rows: list[dict[str, Any]] = []
current_results = {}
for label, spec in {**reference_specs, **candidate_specs}.items():
    result = build_result(label, CURRENT_INPUTS, spec)
    current_results[label] = result
    rows.append(metric_row(label, result, spec, "current"))

current_grid = pd.DataFrame(rows)
current_grid = current_grid.sort_values(["is_reference", "target_loss", "max_dd", "relative_return"], ascending=[False, True, True, False])
current_grid.to_csv(CARD_DIR / "grid_results.csv", index=False)

pass_count = int((current_grid[~current_grid["is_reference"]]["target_pass"]).sum())
relative_pass = current_grid[(~current_grid["is_reference"]) & (current_grid["relative_return"] >= TARGET_RELATIVE_RETURN)]
low_dd = current_grid[(~current_grid["is_reference"])].sort_values(["max_dd", "relative_return"], ascending=[True, False]).head(10)
rel_pass_low_dd = relative_pass.sort_values(["max_dd", "relative_return"], ascending=[True, False]).head(10)

print(f"current_candidate_count={len(candidate_specs)}")
print(f"current_target_pass_count={pass_count}")
print("\nLowest MaxDD candidates:")
display(low_dd[["label", "relative_return", "max_dd", "return", "avg_pos", "drawdown_control_days"]])
print("\nLowest MaxDD candidates that still keep >=30pp relative return:")
display(rel_pass_low_dd[["label", "relative_return", "max_dd", "return", "avg_pos", "drawdown_control_days"]])


current_candidate_count=360
current_target_pass_count=0

Lowest MaxDD candidates:


,label,relative_return,max_dd,return,avg_pos,drawdown_control_days
22,global_s0.06_f0.10_m0.00,-689.407679,10.658739,-9.043962,0.004771,1558
42,global_s0.08_f0.10_m0.00,-689.407679,10.658739,-9.043962,0.004771,1557
2,global_s0.04_f0.10_m0.00,-690.234720,11.471097,-9.871003,0.004527,1562
7,global_s0.04_f0.14_m0.00,-692.836411,14.026599,-12.472694,0.004794,1562
27,global_s0.06_f0.14_m0.00,-693.566906,14.744125,-13.203189,0.005011,1558
47,global_s0.08_f0.14_m0.00,-693.566906,14.744125,-13.203189,0.005011,1557
62,global_s0.10_f0.14_m0.00,-693.566906,14.744125,-13.203189,0.005011,1554
77,global_s0.12_f0.14_m0.00,-693.566906,14.744125,-13.203189,0.005011,1551
3,global_s0.04_f0.10_m0.05,-684.478258,17.013272,-4.114541,0.068226,1545
43,global_s0.08_f0.10_m0.05,-692.390455,17.435568,-12.026738,0.065241,1511



Lowest MaxDD candidates that still keep >=30pp relative return:


,label,relative_return,max_dd,return,avg_pos,drawdown_control_days
260,panic_break_s0.12_f0.14_m0.15,641.691145,32.12018,1322.054862,0.433658,173
214,panic_break_s0.06_f0.18_m0.10,595.590032,32.12018,1275.953749,0.432220,215
124,anchor_break_s0.06_f0.18_m0.10,67.574683,32.12018,747.938400,0.405849,454
153,anchor_break_s0.10_f0.14_m0.05,46.679801,32.12018,727.043518,0.383778,412
204,panic_break_s0.06_f0.10_m0.10,658.068010,32.12018,1338.431727,0.420303,224
114,anchor_break_s0.06_f0.10_m0.10,160.090427,32.12018,840.454144,0.386046,465
237,panic_break_s0.08_f0.22_m0.00,738.647121,32.12018,1419.010838,0.440014,186
225,panic_break_s0.08_f0.10_m0.15,521.511078,32.12018,1201.874795,0.426811,222
185,panic_break_s0.04_f0.10_m0.15,484.397051,32.12018,1164.760769,0.426859,227
217,panic_break_s0.06_f0.22_m0.00,1015.768581,32.12018,1696.132298,0.441152,199


In [6]:
selected_labels = list(rel_pass_low_dd.head(5)["label"])
best_target_loss_label = current_grid[~current_grid["is_reference"]].sort_values(["target_loss", "max_dd", "relative_return"], ascending=[True, True, False]).iloc[0]["label"]
if best_target_loss_label not in selected_labels:
    selected_labels.append(str(best_target_loss_label))
for reference_label in reference_specs:
    if reference_label not in selected_labels:
        selected_labels.append(reference_label)

walk_rows: list[dict[str, Any]] = []
selected_payloads: dict[str, Any] = {}
for label in selected_labels:
    spec = reference_specs.get(label) or candidate_specs[label]
    result = current_results[label]
    write_selected_artifacts(label, result, "current")
    selected_payloads[f"current::{label}"] = result.payload
    for wf_label, inputs in WF_INPUTS.items():
        wf_result = build_result(label, inputs, spec)
        write_selected_artifacts(label, wf_result, wf_label)
        selected_payloads[f"{wf_label}::{label}"] = wf_result.payload
        walk_rows.append(metric_row(label, wf_result, spec, wf_label))

walkforward = pd.DataFrame(walk_rows)
walkforward.to_csv(CARD_DIR / "walkforward_results.csv", index=False)

wf_summary = (
    walkforward.groupby("label")
    .agg(
        wf_windows=("window", "count"),
        wf_target_pass_count=("target_pass", "sum"),
        worst_wf_max_dd=("max_dd", "max"),
        min_wf_relative_return=("relative_return", "min"),
        avg_wf_return=("return", "mean"),
        avg_wf_pos=("avg_pos", "mean"),
    )
    .reset_index()
    .sort_values(["wf_target_pass_count", "worst_wf_max_dd", "min_wf_relative_return"], ascending=[False, True, False])
)
wf_summary.to_csv(CARD_DIR / "walkforward_summary.csv", index=False)

print("selected_labels:", selected_labels)
print("\nWalk-forward summary:")
display(wf_summary)


selected_labels: ['panic_break_s0.12_f0.14_m0.15', 'panic_break_s0.06_f0.18_m0.10', 'anchor_break_s0.06_f0.18_m0.10', 'anchor_break_s0.10_f0.14_m0.05', 'panic_break_s0.06_f0.10_m0.10', 'anchor_confirm_no_dd_control', 'global_guard_default']

Walk-forward summary:


,label,wf_windows,wf_target_pass_count,worst_wf_max_dd,min_wf_relative_return,avg_wf_return,avg_wf_pos
3,global_guard_default,3,0,28.254676,-15.413925,21.411235,0.260753
2,anchor_confirm_no_dd_control,3,0,30.183184,55.796392,50.467149,0.314751
4,panic_break_s0.06_f0.10_m0.10,3,0,30.183184,-17.670623,21.749837,0.248766
1,anchor_break_s0.10_f0.14_m0.05,3,0,30.183184,-30.492412,18.666948,0.225718
0,anchor_break_s0.06_f0.18_m0.10,3,0,30.183184,-36.884994,17.128480,0.253679
5,panic_break_s0.06_f0.18_m0.10,3,0,31.257400,-31.173411,15.809758,0.260736
6,panic_break_s0.12_f0.14_m0.15,3,0,31.958154,-25.752013,15.194544,0.261496


In [7]:
current_by_label = current_grid.set_index("label")
wf_by_label = wf_summary.set_index("label")
current_target_pass_count = int((current_grid[~current_grid["is_reference"]]["target_pass"]).sum())
wf_all_target_pass_labels = [
    label
    for label, group in walkforward.groupby("label")
    if len(group) == len(WF_INPUTS) and bool(group["target_pass"].all())
]
strict_relative_pass_min_dd = rel_pass_low_dd.iloc[0].to_dict() if not rel_pass_low_dd.empty else None
lowest_dd_candidate = low_dd.iloc[0].to_dict() if not low_dd.empty else None

selection = {
    "card": CARD,
    "decision": "diagnostic_no_promotion",
    "target": {
        "relative_return_min_pp": TARGET_RELATIVE_RETURN,
        "max_dd_max_pct": TARGET_MAX_DD,
    },
    "candidate_count": int(len(candidate_specs)),
    "current_target_pass_count": current_target_pass_count,
    "wf_all_target_pass_labels": wf_all_target_pass_labels,
    "strict_relative_pass_min_dd": strict_relative_pass_min_dd,
    "lowest_dd_candidate": lowest_dd_candidate,
    "selected_labels": selected_labels,
    "artifacts": {
        "notebook": str((CARD_DIR / f"{CARD}.ipynb").relative_to(ROOT)),
        "grid_results": str((CARD_DIR / "grid_results.csv").relative_to(ROOT)),
        "walkforward_results": str((CARD_DIR / "walkforward_results.csv").relative_to(ROOT)),
        "walkforward_summary": str((CARD_DIR / "walkforward_summary.csv").relative_to(ROOT)),
        "selected_artifacts_dir": str(ARTIFACT_DIR.relative_to(ROOT)),
    },
    "interpretation": [
        "No regime-aware equity drawdown governor passed the strict current target of >=30pp excess and <=15% MaxDD.",
        "The only configurations below 15% current MaxDD were always-on/global near-zero exposure branches with deeply negative relative return, which is underparticipation rather than risk improvement.",
        "Feature-filtered governors preserved large excess return but left current MaxDD around the low-30% floor, so this overlay cannot solve the 15% target alone.",
        "Do not keep tightening this execution overlay. The next aligned loop should move drawdown/rebound trade-off upstream into actor training or model selection with path-level drawdown and rebound opportunity-cost terms, then re-run fixed-split plus walk-forward gates.",
    ],
}
def json_safe(value):
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if pd.isna(value) if isinstance(value, (float, int)) else False:
        return None
    return value


with (CARD_DIR / "selection.json").open("w") as f:
    json.dump(json_safe(selection), f, indent=2, allow_nan=False)

summary_lines = [
    f"# {CARD} conclusion",
    "",
    f"- Current grid candidates: `{len(candidate_specs)}`",
    f"- Strict current target pass count: `{current_target_pass_count}`",
    f"- Walk-forward all-window target-pass labels: `{wf_all_target_pass_labels}`",
]
if lowest_dd_candidate:
    summary_lines.append(
        "- Lowest current MaxDD branch: "
        f"`{lowest_dd_candidate['label']}` with MaxDD `{lowest_dd_candidate['max_dd']:.2f}%`, "
        f"relative `{lowest_dd_candidate['relative_return']:.2f}pp`, avg_pos `{lowest_dd_candidate['avg_pos']:.3f}`."
    )
if strict_relative_pass_min_dd:
    summary_lines.append(
        "- Best current drawdown among branches that keep `>=30pp` excess: "
        f"`{strict_relative_pass_min_dd['label']}` with MaxDD `{strict_relative_pass_min_dd['max_dd']:.2f}%`, "
        f"relative `{strict_relative_pass_min_dd['relative_return']:.2f}pp`, avg_pos `{strict_relative_pass_min_dd['avg_pos']:.3f}`."
    )
summary_lines.extend(["", "## Decision", ""])
summary_lines.extend([f"- {line}" for line in selection["interpretation"]])
summary_lines.extend([
    "",
    "## Next Experiment",
    "",
    "- Stop trying to reach `15%` MaxDD with a post-hoc execution cap on this actor pair.",
    "- Create a notebook-first reward/model-selection training loop that optimizes path-level drawdown while explicitly penalizing missed rebound participation, then evaluate saved walk-forward actors before any runtime promotion.",
])
conclusion_md = "\n".join(summary_lines)
display(Markdown(conclusion_md))


# regime_aware_equity_drawdown_guard_v1 conclusion

- Current grid candidates: `360`
- Strict current target pass count: `0`
- Walk-forward all-window target-pass labels: `[]`
- Lowest current MaxDD branch: `global_s0.06_f0.10_m0.00` with MaxDD `10.66%`, relative `-689.41pp`, avg_pos `0.005`.
- Best current drawdown among branches that keep `>=30pp` excess: `panic_break_s0.12_f0.14_m0.15` with MaxDD `32.12%`, relative `641.69pp`, avg_pos `0.434`.

## Decision

- No regime-aware equity drawdown governor passed the strict current target of >=30pp excess and <=15% MaxDD.
- The only configurations below 15% current MaxDD were always-on/global near-zero exposure branches with deeply negative relative return, which is underparticipation rather than risk improvement.
- Feature-filtered governors preserved large excess return but left current MaxDD around the low-30% floor, so this overlay cannot solve the 15% target alone.
- Do not keep tightening this execution overlay. The next aligned loop should move drawdown/rebound trade-off upstream into actor training or model selection with path-level drawdown and rebound opportunity-cost terms, then re-run fixed-split plus walk-forward gates.

## Next Experiment

- Stop trying to reach `15%` MaxDD with a post-hoc execution cap on this actor pair.
- Create a notebook-first reward/model-selection training loop that optimizes path-level drawdown while explicitly penalizing missed rebound participation, then evaluate saved walk-forward actors before any runtime promotion.